#  Advanced Multi-Metric Character Consistency Pipeline — Google Colab Edition

Verify visual and semantic character consistency across generated assets and panels using HSV color correlation, SSIM, Canny Edge, Style Gram, CLIP, DINOv2, and aesthetic scoring.

 **Runtime Requirement**: Select **T4 GPU** (or any GPU) to execute.


### Setup Step: Prepare Environment
Select your setup mode to either **Clone Repository** directly from GitHub (recommended) or **Upload ZIP** archive.

In [ ]:
#@title Choose Setup Method { run: "auto" }
SETUP_MODE = "git" #@param ["git", "zip"]

import os, subprocess, zipfile
from google.colab import files

if SETUP_MODE == "git":
    REPO_URL   = "https://github.com/Cyberpunk-San/Indie-Comic.git"
    REPO_DIR   = "/content/indie_comic_pipeline"
    SUBDIR     = "indie_comic_pipeline"

    if not os.path.exists(REPO_DIR):
        print(f" Cloning repo from {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        print(" Repository already present — pulling latest changes...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    PIPELINE_ROOT = os.path.join(REPO_DIR, SUBDIR)
    os.chdir(PIPELINE_ROOT)
    print(f" Working directory set to: {os.getcwd()}")

    for d in ["outputs/fusion", "outputs/comics", "outputs/characters"]:
        os.makedirs(d, exist_ok=True)
    print(" Directory structure ready.")
else:
    print(" Upload your indie_comic_pipeline.zip file:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            print(f" Unzipped: {filename}")
            break

    %cd /content/indie_comic_pipeline
    print(f" Current working directory: {os.getcwd()}")

### Setup Step: Self-Healing Hotfixes
Automatically audits and patches any duplicate imports, missing variables, or consistency checker reference setup issues in the pipeline scripts.

In [ ]:
# Programmatic Hotfix Applier
import os

def apply_hotfixes():
    print(" Auditing files for known runtime bugs...")
    
    # Fix 1: langchain_code/fusion_engine.py numpy name issue
    f_path = "langchain_code/fusion_engine.py"
    if os.path.exists(f_path):
        with open(f_path, "r", encoding="utf-8") as f:
            content = f.read()
        if "import numpy as np" not in content[:300]:
            print("  - Patching langchain_code/fusion_engine.py to add numpy import at top")
            content = content.replace("import numpy as np", "")
            content = "import numpy as np\n" + content
            with open(f_path, "w", encoding="utf-8") as f:
                f.write(content)
    
    # Fix 2: Set reference in generate_panels/components consistency checks
    for root, dirs, files in os.walk("."):
        for file in files:
            if file in ["generate_panels.py", "generate_components.py"]:
                path = os.path.join(root, file)
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                if "checker = get_consistency_checker()" in content and "checker.set_reference" not in content:
                    print(f"  - Patching {path}: adding missing set_reference call")
                    content = content.replace(
                        "checker = get_consistency_checker()",
                        "checker = get_consistency_checker()\n        if os.path.exists(ref_path):\n            checker.set_reference(ref_path)"
                    )
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(content)
                        
    print(" Hotfix audit complete. All scripts are ready and bug-free!")

apply_hotfixes()

### Step 1: Install Dependencies
Installs required libraries including PyTorch with GPU compatibility, diffusers, accelerate, langchain, and metrics libraries.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors langchain-ollama langchain-core pyyaml opencv-python-headless pillow scikit-image peft torchmetrics torchvision matplotlib pandas torchao>=0.16.0

### Step 3: Configure Settings for Colab GPU
Update `config/settings.yaml` dynamically with GPU device parameters and setup target story variables.

In [ ]:
# ============================================================
#    PIPELINE CONFIGURATION  —  Edit these values
# ============================================================
CHARACTER_NAME = "Spider-Man"        # Any fictional character
STORY_WORLD    = "Cyberpunk 2077"    # Any story / universe / setting
PAGE_TO_RENDER = 1                  # Which page to render panels for (1-10)
IMG_WIDTH      = 1024                # Resolution width (must be multiple of 8, max 1024)
IMG_HEIGHT     = 1024                # Resolution height
INFERENCE_STEPS = 30                # Higher = better, lower = faster (default: 30)
GUIDANCE_SCALE = 7.5                # How closely to follow prompts
SEED           = 42                 # Reprod seed
OLLAMA_MODEL   = "llama3.2"

import yaml, os

with open('config/settings.yaml', 'r') as f:
    settings = yaml.safe_load(f)

# Configure settings for GPU execution of SDXL, SD v1.5 and LoRA
settings['models']['sdxl']['device'] = 'cuda'
settings['models']['sdxl']['name'] = 'stabilityai/stable-diffusion-xl-base-1.0'
settings['models']['sd15']['device'] = 'cuda'
settings['models']['sd15']['name'] = 'runwayml/stable-diffusion-v1-5'
settings['models']['lora']['name'] = 'artificialguybr/LineAniRedmond-LinearMangaSDXL-V2'
settings['models']['lora']['trigger_words'] = 'LineAniAF, lineart'
settings['generation']['default_size']['width'] = IMG_WIDTH
settings['generation']['default_size']['height'] = IMG_HEIGHT
settings['generation']['inference_steps'] = INFERENCE_STEPS
settings['generation']['guidance_scale'] = GUIDANCE_SCALE
settings['generation']['seed'] = SEED
settings['langchain']['model'] = OLLAMA_MODEL
settings['langchain']['ollama_url'] = 'http://localhost:11434'

with open('config/settings.yaml', 'w') as f:
    yaml.safe_dump(settings, f)
    
print(f" settings.yaml patched with: {CHARACTER_NAME} × {STORY_WORLD} | Steps={INFERENCE_STEPS} | cuda=Active")

### Step 6.1: Advanced Consistency Suite Validation
Evaluates character and asset visual coherence against the anchor reference using **8 specific mathematical & neural metrics**:

1. **Color Consistency**: Hue & Saturation HSV space correlation ($d(H_1, H_2)$)
2. **SSIM**: Structural Similarity Index (luminance, contrast, structural mapping)
3. **Canny Edge Density**: Compares drawing stroke active pixels ratio
4. **Art Style Gram Matrix**: Texture texture style correlations on Sobel gradients
5. **CLIP Image Similarity**: 512-dimensional semantic visual visual validation
6. **DINOv2 Feature Similarity**: High-fidelity spatial transformer representation alignment
7. **Offline Aesthetic Score**: Combines colorfulness, contrast, and sharpness locally
8. **Grayscale Correlation**: Legacy structure coefficient benchmark

In [ ]:
import sys, os
sys.path.append(os.getcwd())
sys.path.append('/content/indie_comic_pipeline')
sys.path.append('/content/Indie-Comic/indie_comic_pipeline')
import glob, pandas as pd
from utils.consistency_checker import get_consistency_checker

checker = get_consistency_checker()
refs = [
    "outputs/characters/character_reference.png",
    "outputs/characters/character_reference_sd15.png",
    "outputs/characters/character_reference_sdxl_lora.png"
]
ref_found = next((r for r in refs if os.path.exists(r)), None)
components = sorted(glob.glob("outputs/comics/component_*.png"))

if ref_found and components:
    print(f" Running full Consistency Suite with anchor: {ref_found}")
    checker.set_reference(ref_found)
    
    results = []
    for path in components:
        res = checker.check_consistency(path)
        results.append({
            "File": os.path.basename(path),
            "Overall Score": f"{res['score']:.3f}",
            "HSV Color Match": f"{res['color_score']:.3f}",
            "SSIM Structure": f"{res['ssim_score']:.3f}",
            "Edge Density Match": f"{res['edge_score']:.3f}",
            "Style Gram Match": f"{res['style_score']:.3f}",
            "Aesthetic Score": f"{res['aesthetic_score']:.3f}",
            "CLIP Semantic": f"{res['clip_img_score']:.3f}" if res['clip_img_score'] is not None else "N/A",
            "DINOv2 Structural": f"{res['dinov2_score']:.3f}" if res['dinov2_score'] is not None else "N/A",
            "Consistent": " Yes" if res['consistent'] else " No"
        })
        
    df_cons = pd.DataFrame(results)
    display(df_cons)
else:
    print(" Missing generated assets or anchor reference sheet. Run step 6 first.")

### Step 7.2: Verify Comic Panels Character Consistency
Runs the Consistency Suite on the generated page panels to verify character visual integrity.

In [ ]:
import sys, os
sys.path.append(os.getcwd())
sys.path.append('/content/indie_comic_pipeline')
sys.path.append('/content/Indie-Comic/indie_comic_pipeline')
import glob, pandas as pd
from utils.consistency_checker import get_consistency_checker

checker = get_consistency_checker()
refs = [
    "outputs/characters/character_reference.png",
    "outputs/characters/character_reference_sd15.png",
    "outputs/characters/character_reference_sdxl_lora.png"
]
ref_found = next((r for r in refs if os.path.exists(r)), None)
panel_paths = sorted(glob.glob(f"outputs/comics/page_{PAGE_TO_RENDER}_panel_*.png"))

if ref_found and panel_paths:
    print(f" Verifying character consistency across panels utilizing anchor ref: {ref_found}")
    checker.set_reference(ref_found)
    
    results = []
    for path in panel_paths:
        res = checker.check_consistency(path)
        results.append({
            "Panel": os.path.basename(path),
            "Consistency score": f"{res['score']:.3f}",
            "HSV Color Match": f"{res['color_score']:.3f}",
            "SSIM structure": f"{res['ssim_score']:.3f}",
            "Edge Density Match": f"{res['edge_score']:.3f}",
            "Style Gram Match": f"{res['style_score']:.3f}",
            "Aesthetic Score": f"{res['aesthetic_score']:.3f}",
            "CLIP image match": f"{res['clip_img_score']:.3f}" if res['clip_img_score'] is not None else "N/A",
            "DINOv2 similarity": f"{res['dinov2_score']:.3f}" if res['dinov2_score'] is not None else "N/A",
            "Consistent": " Yes" if res['consistent'] else " No"
        })
    df_pan_cons = pd.DataFrame(results)
    display(df_pan_cons)
else:
    print(" No panel images found or reference sheet missing.")

### Step 9: Download All Generated Outputs
Creates a consolidated ZIP archive of all output files and triggers the browser download.

In [ ]:
import os, zipfile
from google.colab import files

ZIP_PATH = "/content/indie_comic_outputs.zip"
print(" Packaging outputs folder into ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("outputs"):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname("outputs"))
            zf.write(fpath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f" ZIP created: {ZIP_PATH} ({size_mb:.1f} MB)")
print("⬇ Initiating browser download...")
files.download(ZIP_PATH)